# Instagram Reels → Bunny Storage → Instagram (Google Colab)

Run the cells from top to bottom.

1. **Setup:** gets the code, installs it and connects Google Drive
2. **Secrets:** loads your Bunny details and Instagram tokens from Colab Secrets (🔑 icon on the left)
3. **Sync:** downloads every reel of a profile into the Bunny folder `<username>/`
4. **Publish:** posts the Bunny videos as Reels to the account that owns the access token, in batches

Only archive or repost videos you own or have the rights to.

## 1. Setup
If the GitHub repo is **private**, first add a Colab secret `GITHUB_TOKEN`
(a GitHub personal access token with read access to the repo).

In [ ]:
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/instagram-reels-to-bunny.git"  #@param {type:"string"}
USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}

import os, subprocess
from google.colab import userdata

url = REPO_URL
try:
    gh_token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{gh_token}@")
except Exception:
    pass  # public repo, no token needed

if not os.path.isdir("/content/instagram-reels-to-bunny"):
    subprocess.run(["git", "clone", "-q", url, "/content/instagram-reels-to-bunny"], check=True)
else:
    subprocess.run(["git", "-C", "/content/instagram-reels-to-bunny", "pull", "-q"], check=True)
%cd /content/instagram-reels-to-bunny
!pip install -q -r requirements.txt
!pip install -q -U yt-dlp
!ffmpeg -version | head -1

# Keep the history file (prevents duplicate posts) and cookies in Google Drive,
# because Colab deletes /content when the runtime stops.
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/reels2bunny"
else:
    DATA_DIR = "/content/reels2bunny-data"
os.makedirs(DATA_DIR, exist_ok=True)
HISTORY = f"{DATA_DIR}/instagram_uploads.json"
print("Data folder:", DATA_DIR)

## 2. Secrets
In Colab open **🔑 Secrets** (left sidebar), add these and switch on *Notebook access*:

| Secret | Value |
|---|---|
| `BUNNY_STORAGE_ZONE` | storage zone name |
| `BUNNY_STORAGE_PASSWORD` | zone password (Storage → zone → FTP & API Access) |
| `BUNNY_STORAGE_REGION` | empty for Falkenstein, or `uk`, `ny`, `la`, `sg`, `se`, `br`, `jh`, `syd` |
| `BUNNY_CDN_HOSTNAME` | pull zone hostname, e.g. `my-reels.b-cdn.net` |
| `IG_TOKEN_1`, `IG_TOKEN_2`, … | one Instagram access token per account you want to post to |

In [ ]:
from google.colab import userdata

for name in ["BUNNY_STORAGE_ZONE", "BUNNY_STORAGE_PASSWORD", "BUNNY_STORAGE_REGION",
             "BUNNY_CDN_HOSTNAME", "BUNNY_BASE_PATH", "BUNNY_API_KEY"]:
    try:
        os.environ[name] = userdata.get(name) or ""
        print(f"✔ {name}")
    except Exception:
        print(f"  {name} not set" + ("  <-- required" if name in
              ("BUNNY_STORAGE_ZONE", "BUNNY_STORAGE_PASSWORD") else ""))

## 3. Sync: download all reels of a profile to Bunny
Instagram needs a logged-in session to list reels. Export `cookies.txt` (Netscape format)
from a browser that's logged in to instagram.com, for example with the "Get cookies.txt LOCALLY"
extension. Upload it when asked; it's saved in your data folder for next time.

In [ ]:
from google.colab import files
COOKIES = f"{DATA_DIR}/cookies.txt"
if not os.path.exists(COOKIES):
    up = files.upload()               # choose your cookies.txt
    with open(COOKIES, "wb") as fh:
        fh.write(next(iter(up.values())))
print("Cookies:", COOKIES)

In [ ]:
USERNAME = "instagram_username"  #@param {type:"string"}
BATCH_SIZE = 20   #@param {type:"integer"}
BATCH_PAUSE = 10  #@param {type:"integer"}
WORKERS = 3       #@param {type:"integer"}
DRY_RUN = True    #@param {type:"boolean"}

flags = "--dry-run" if DRY_RUN else ""
!python -m reels2bunny sync -u "$USERNAME" -c "$COOKIES" --batch-size $BATCH_SIZE --batch-pause $BATCH_PAUSE --workers $WORKERS $flags

## 4. Publish: post Bunny videos as Reels to your account
**The access token decides which account gets the posts.** Choose which token secret to use.
`EXPECT_USERNAME` is a safety check: the run stops before posting anything if the token
belongs to a different account.

Keep `DRY_RUN` ticked the first time. It shows the target account, the videos, the
batches and the caption, and posts nothing.

In [ ]:
TOKEN_SECRET = "IG_TOKEN_1"        #@param {type:"string"}
EXPECT_USERNAME = ""               #@param {type:"string"}
FOLDER = "instagram_username"      #@param {type:"string"}
CAPTION = "{original}"             #@param {type:"string"}
BATCH_SIZE = 10                    #@param {type:"integer"}
WORKERS = 2                        #@param {type:"integer"}
BATCH_PAUSE = 60                   #@param {type:"integer"}
LIMIT = 0                          #@param {type:"integer"}
DRY_RUN = True                     #@param {type:"boolean"}

import shlex
os.environ["IG_ACCESS_TOKEN"] = userdata.get(TOKEN_SECRET)   # token stays out of the output
args = ["publish", "-f", FOLDER, "--caption", CAPTION, "--history", HISTORY,
        "--batch-size", str(BATCH_SIZE), "--workers", str(WORKERS),
        "--batch-pause", str(BATCH_PAUSE)]
if EXPECT_USERNAME: args += ["--expect-username", EXPECT_USERNAME]
if LIMIT: args += ["--limit", str(LIMIT)]
if DRY_RUN: args.append("--dry-run")
cmd = "python -m reels2bunny " + " ".join(shlex.quote(a) for a in args)
!{cmd}

## 5. Links of published Reels

In [ ]:
ACCOUNT = ""   #@param {type:"string"}
acc = f'--account "{ACCOUNT}"' if ACCOUNT else ""
!python -m reels2bunny links --history "$HISTORY" --format csv $acc

### Tips
- **Colab stops after a few hours or when idle.** Just run the cells again. Finished reels and
  published posts are skipped because progress is saved in Bunny and in the Drive history file.
- **Exit code 3 / "publishing limit"** means the account used its 24h API post quota. Run
  the publish cell again tomorrow.
- **"not logged in / blocked"** during sync: delete `cookies.txt` from the data folder,
  export fresh cookies and run the cookie cell again.